# House Price Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `price`

## 2 · Project Overview

We predict **house prices** based on structural features (square footage, bedrooms, bathrooms, age, garage), lot size, neighborhood type, and property condition.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a house's square footage, bedrooms, bathrooms, age, lot size, garage capacity, neighborhood, and condition rating, predict its sale price.

## 5 · Why This Project Matters

- **Real estate valuation** is a fundamental financial prediction problem.
- Automated pricing supports buyers, sellers, agents, and lenders.
- Understanding price drivers helps investment decisions.
- Classic regression dataset with mixed numeric and categorical features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | sqft, bedrooms, bathrooms, age_years, lot_sqft, garage_cars, neighborhood, condition |
| **Target** | price (continuous, USD) |
| **Range** | ~$30K – $2M |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "price"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: price
Save dir: E:\Github\Machine-Learning-Projects\Regression\House Price Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 houses with structural and location features.

In [4]:
np.random.seed(SEED)
n = 8000
sqft = np.random.randint(600, 5000, n)
bedrooms = np.random.randint(1, 7, n)
bathrooms = (bedrooms * 0.6 + np.random.choice([0, 0.5, 1], n)).clip(1, 5)
age_years = np.random.randint(0, 80, n)
lot_sqft = (sqft * np.random.uniform(1.5, 5, n)).astype(int)
garage_cars = np.random.choice([0, 1, 2, 3], n, p=[0.15, 0.35, 0.35, 0.15])
neighborhood = np.random.choice(["Downtown", "Suburban", "Rural", "Waterfront"], n,
                                 p=[0.25, 0.4, 0.2, 0.15])
nbr_bonus = np.where(neighborhood == "Waterfront", 80000,
            np.where(neighborhood == "Downtown", 40000,
            np.where(neighborhood == "Suburban", 15000, 0)))
condition = np.random.choice(["Poor", "Fair", "Good", "Excellent"], n, p=[0.1, 0.25, 0.4, 0.25])
cond_mult = np.where(condition == "Poor", 0.75, np.where(condition == "Fair", 0.9,
            np.where(condition == "Good", 1.0, 1.15)))

price = (80 * sqft + 15000 * bedrooms + 12000 * bathrooms
         - 600 * age_years + 5 * lot_sqft + 20000 * garage_cars
         + nbr_bonus) * cond_mult
price = np.round(price + np.random.normal(0, 15000, n), -2).clip(30000, 2000000)

df = pd.DataFrame({"sqft": sqft, "bedrooms": bedrooms, "bathrooms": bathrooms,
                    "age_years": age_years, "lot_sqft": lot_sqft, "garage_cars": garage_cars,
                    "neighborhood": neighborhood, "condition": condition, "price": price})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['price'].describe()}")
df.head()

Dataset shape: (8000, 9)
Target stats:
count      8000.000000
mean     385051.737500
std      140466.803871
min       30900.000000
25%      275175.000000
50%      381450.000000
75%      487000.000000
max      831000.000000
Name: price, dtype: float64


,sqft,bedrooms,bathrooms,age_years,lot_sqft,garage_cars,neighborhood,condition,price
0,1460,2,1.2,32,2667,1,Downtown,Good,216000.0
1,4372,1,1.0,78,7198,2,Suburban,Fair,398900.0
2,3692,4,3.4,34,12051,2,Suburban,Poor,389500.0
3,1066,6,3.6,64,1821,1,Suburban,Fair,182000.0
4,4044,6,4.6,62,11621,1,Suburban,Fair,467800.0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

DATA VALIDATION

Shape: (8000, 9)

Missing values:
sqft            0
bedrooms        0
bathrooms       0
age_years       0
lot_sqft        0
garage_cars     0
neighborhood    0
condition       0
price           0
dtype: int64

Duplicate rows: 0

Dtypes:
sqft              int32
bedrooms          int32
bathrooms       float64
age_years         int32
lot_sqft          int64
garage_cars       int64
neighborhood     object
condition        object
price           float64
dtype: object

Target 'price' confirmed.

Target stats:
count      8000.000000
mean     385051.737500
std      140466.803871
min       30900.000000
25%      275175.000000
50%      381450.000000
75%      487000.000000
max      831000.000000
Name: price, dtype: float64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["sqft", "bedrooms", "bathrooms", "age_years", "lot_sqft", "garage_cars"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    ax.scatter(df[col], df[TARGET], alpha=0.2, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel(TARGET)
    ax.set_title(f"{col} vs {TARGET}")
plt.tight_layout()
plt.show()

corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `price`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Price ($)")

axes[1].boxplot([df[df["neighborhood"]==nb][TARGET] for nb in ["Rural","Suburban","Downtown","Waterfront"]],
                labels=["Rural","Suburban","Downtown","Waterfront"])
axes[1].set_title("Price by Neighborhood")
axes[1].set_ylabel("Price ($)")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Std: ${df[TARGET].std():,.0f}, Skew: {df[TARGET].skew():.3f}")

Range: $30,900 to $831,000
Mean: $385,052, Median: $381,450
Std: $140,467, Skew: 0.193


## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

Train: (6400, 8), Test: (1600, 8)
Target — Train mean: 385,256.48, Test mean: 384,232.75


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `neighborhood`, `condition` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Price naturally right-skewed — acceptable for tree models.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["price_per_sqft_proxy"] = X_train["sqft"] / (X_train["bedrooms"] + 1)
X_test["price_per_sqft_proxy"] = X_test["sqft"] / (X_test["bedrooms"] + 1)

X_train["total_rooms"] = X_train["bedrooms"] + X_train["bathrooms"]
X_test["total_rooms"] = X_test["bedrooms"] + X_test["bathrooms"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (10): ['sqft', 'bedrooms', 'bathrooms', 'age_years', 'lot_sqft', 'garage_cars', 'neighborhood', 'condition', 'price_per_sqft_proxy', 'total_rooms']


## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

BASELINE: Linear Regression
RMSE : 45,974.04
MAE  : 36,968.62
R2   : 0.8899


## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Regressors:
                               Adjusted R-Squared  R-Squared          RMSE  Time Taken
Model                                                                                 
CatBoostRegressor                        0.987208   0.987288  15618.819079    2.103897
LGBMRegressor                            0.984423   0.984520  17235.564533    0.087844
HistGradientBoostingRegressor            0.983956   0.984056  17491.870721    0.303969
GradientBoostingRegressor                0.979995   0.980120  19532.305920    0.878544
XGBRegressor                             0.979608   0.979735  19720.134339    0.210420
ExtraTreesRegressor                      0.971785   0.971962  23196.196568    1.621283
RandomForestRegressor                    0.965779   0.965993  25546.152910    2.489392
BaggingRegressor                         0.959437   0.959690  27812.838490    0.244993
KNeighborsRegressor                      0.936639   0.937036  34760.692822    0.074987
ExtraTreeR

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

FLAML best model: catboost
RMSE: 16,146.97
R2  : 0.9864


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost RMSE: 15,801.12  (1.8s)


Training until validation scores don't improve for 30 rounds


Did not meet early stopping. Best iteration is:
[300]	valid_0's l2: 2.80297e+08
LightGBM RMSE: 16,742.08  (9.3s)


XGBoost failed: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by RMSE):
                       RMSE       MAE      R2
CatBoost           15801.12  12649.76  0.9870
FLAML              16146.97  12895.55  0.9864
LightGBM           16742.08  13397.38  0.9854
Linear Regression  45974.04  36968.62  0.8899

Top 3 models: ['CatBoost', 'FLAML', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")


Detailed Metrics — Top 3:

  CatBoost:
    RMSE : 15,801.12
    MAE  : 12,649.76
    R2   : 0.9870

  FLAML:
    RMSE : 16,146.97
    MAE  : 12,895.55
    R2   : 0.9864

  LightGBM:
    RMSE : 16,742.08
    MAE  : 13,397.38
    R2   : 0.9854


## 24 · Error Analysis

Examine residuals from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

Residual stats (CatBoost):
  Mean: -249.58, Std: 15,799.15
  Min: -51,747.86, Max: 51,870.45
  Median: -348.52


## 25 · Interpretation and Business Insight

**Key findings:**
- **Square footage** is the strongest price predictor (r ~ 0.85+).
- **Neighborhood** (Waterfront, Downtown) adds significant premium.
- **Age** negatively impacts price, modulated by condition.
- **Garage capacity** has a strong linear effect (~$20K per space).
- **Condition** acts as a multiplier on base value.

**Business takeaway:** Price is primarily driven by size and location. Renovation (condition improvement) provides direct ROI.

## 26 · Limitations

1. Synthetic data with simplified pricing dynamics.
2. No school district, crime rate, or walkability scores.
3. Missing renovation/remodel history.
4. No time dimension (market conditions).
5. Simplified neighborhoods (4 types).

## 27 · How to Improve This Project

1. Use real MLS/Zillow data with more features.
2. Add geographic coordinates and spatial features.
3. Include school ratings, crime stats, walkability.
4. Model temporal price trends.
5. Add comparable sales (comps) features.

## 28 · Production Considerations

- Deploy as an automated valuation model (AVM).
- Output prediction intervals for risk assessment.
- Update quarterly with new sales data.
- Monitor for market regime changes.
- Segment by neighborhood for more accurate predictions.

## 29 · Common Mistakes

1. Not log-transforming skewed prices for linear models.
2. Ignoring location — it's the #1 real estate factor.
3. Using RMSE alone without MAE for outlier sensitivity.
4. Not considering multicollinearity (sqft vs. bedrooms).
5. Overfitting on small neighborhoods.

## 30 · Mini Challenge / Exercises

1. Log-transform `price` — does R² improve for Linear Regression?
2. Remove `neighborhood` — how much does RMSE increase?
3. Plot residuals by neighborhood to check for heteroscedasticity.
4. Create `sqft_per_bedroom` and test its impact.
5. Try Ridge/Lasso to identify the most penalized features.

## 31 · Final Summary / Key Takeaways

1. **Square footage** dominates house price prediction.
2. **Location** (neighborhood) provides the strongest categorical signal.
3. **Age and condition** interact to determine value deterioration/preservation.
4. **Garage capacity** has a clear linear premium.
5. **Tree models** handle the feature interactions naturally.
6. **Prediction intervals** are critical for real estate decisions.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Regression\House Price Prediction\metrics.json
{
  "CatBoost": {
    "rmse": 15801.12,
    "mae": 12649.76,
    "r2": 0.987
  },
  "LightGBM": {
    "rmse": 16742.08,
    "mae": 13397.38,
    "r2": 0.9854
  },
  "Linear Regression": {
    "rmse": 45974.04,
    "mae": 36968.62,
    "r2": 0.8899
  },
  "FLAML": {
    "rmse": 16146.97,
    "mae": 12895.55,
    "r2": 0.9864
  }
}
